## Hybrid Recommendation System

### 1. Objective

Previous experiments evaluated several recommendation strategies independently:

- recent popularity;
- ALS collaborative filtering;
- item-item collaborative filtering;
- content-based recommendation.

These models capture different recommendation signals and show substantially different standalone performance.

The objective of this notebook is to determine whether combining their candidate sets improves recommendation coverage and final Top-12 performance.

The analysis focuses on:

1. candidate recall for each recommendation source;
2. overlap and complementary retrieval between sources;
3. incremental contribution of each source;
4. construction of a combined candidate pool;
5. simple rank fusion for producing hybrid Top-12 recommendations.

Only recommendation sources that provide measurable incremental value will be retained in the final architecture.

In [2]:
import numpy as np
import pandas as pd

In [3]:
transactions = pd.read_csv(
    "../data/raw/transactions_train.csv",
    usecols=["t_dat", "customer_id", "article_id"]
)

transactions["t_dat"] = pd.to_datetime(
    transactions["t_dat"]
)

max_date = transactions["t_dat"].max()
eval_start = max_date - pd.Timedelta(days=6)

train = transactions[
    transactions["t_dat"] < eval_start
].copy()

evaluation = transactions[
    transactions["t_dat"] >= eval_start
].copy()

In [4]:
ground_truth = (
    evaluation
    .groupby("customer_id")["article_id"]
    .apply(set)
    .reset_index(name="actual_articles")
)